# 🎌 MagiV2 Manga Translator — Google Colab Setup

## Trước khi bắt đầu:
1. **Runtime → Change runtime type → T4 GPU** (free) hoặc A100 (Colab Pro)
2. Đảm bảo đã kết nối Google Drive nếu muốn lưu kết quả
3. Cần có **API key** cho LLM (Gemini free hoặc OpenRouter)

---

## 📦 BƯỚC 1: Clone Project & Cài Dependencies

In [ ]:
# ===== ĐIỀN THÔNG TIN CỦA BẠN VÀO ĐÂY =====
GITHUB_REPO = "your-username/your-repo"  # VD: "nguyenvan/manga-translator"
GITHUB_TOKEN = ""  # Để trống nếu repo public, điền token nếu private

# LLM API (chọn 1 trong 2)
GEMINI_API_KEY = ""        # Lấy free tại: https://aistudio.google.com/
OPENROUTER_API_KEY = ""    # Lấy free tại: https://openrouter.ai/ (có model free)

# PostgreSQL (Colab dùng local SQLite thay thế, hoặc neon.tech free)
POSTGRES_HOST = "localhost"   # Đổi thành host của bạn nếu dùng cloud DB
POSTGRES_USER = "magiv2"
POSTGRES_PASSWORD = "magiv2pass"
POSTGRES_DB = "magiv2_db"
POSTGRES_PORT = "5432"
# ============================================

In [ ]:
import os

# Clone repo
if GITHUB_TOKEN:
    repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
else:
    repo_url = f"https://github.com/{GITHUB_REPO}.git"

!git clone {repo_url} /content/magiv2
%cd /content/magiv2
print("✅ Cloned successfully!")

In [ ]:
# Kiểm tra GPU
!nvidia-smi
import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
%%bash
# Cài dependencies — bỏ triton (không cần trên Colab T4)
pip install -q \
  fastapi==0.110.0 \
  uvicorn==0.27.1 \
  sqlalchemy==2.0.23 \
  alembic==1.13.1 \
  asyncpg==0.29.0 \
  psycopg2-binary==2.9.9 \
  openai==1.106.1 \
  google-generativeai \
  transformers==4.39.1 \
  timm==1.0.19 \
  einops==0.8.1 \
  albumentations==2.0.8 \
  omegaconf==2.3.0 \
  onnxruntime-gpu==1.22.0 \
  httpx==0.28.1 \
  python-multipart==0.0.9 \
  python-dotenv==1.1.1 \
  pydantic==2.11.7 \
  pydantic-settings \
  pillow==11.3.0 \
  opencv-python-headless==4.12.0.88 \
  langdetect==1.0.9 \
  langcodes==3.5.0 \
  scikit-image==0.25.2 \
  scikit-learn==1.7.1 \
  PuLP==3.2.2 \
  pyclipper==1.3.0.post6 \
  shapely==2.1.1 \
  sentencepiece==0.2.1 \
  freetype-py==2.5.1 \
  pyvi==0.1.1 \
  pydensecrf@git+https://github.com/lucasb-eyer/pydensecrf.git

echo "✅ Dependencies installed!"

## 🔧 BƯỚC 2: Cấu hình Environment

In [ ]:
# Tạo file .env
import os

# Quyết định dùng LLM nào
if GEMINI_API_KEY:
    llm_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
    llm_api_key = GEMINI_API_KEY
    llm_model = "gemini-2.5-flash"
    llm_tokenizer = "google/gemma-2-2b"
elif OPENROUTER_API_KEY:
    llm_base_url = "https://openrouter.ai/api/v1"
    llm_api_key = OPENROUTER_API_KEY
    llm_model = "google/gemma-3-27b-it:free"  # Model free trên OpenRouter
    llm_tokenizer = "google/gemma-3-27b-it"
else:
    raise ValueError("❌ Cần ít nhất 1 API key: GEMINI_API_KEY hoặc OPENROUTER_API_KEY")

env_content = f"""# Auto-generated for Google Colab
VLLM_BASE_URL={llm_base_url}
VLLM_API_KEY={llm_api_key}
VLLM_MODEL_NAME={llm_model}
VLLM_TOKENIZER={llm_tokenizer}
MAX_TOTAL_TOKENS=32768
GEMINI_API_KEY={GEMINI_API_KEY}

POSTGRES_USER={POSTGRES_USER}
POSTGRES_PASSWORD={POSTGRES_PASSWORD}
POSTGRES_DB={POSTGRES_DB}
POSTGRES_PORT={POSTGRES_PORT}
POSTGRES_HOST={POSTGRES_HOST}

BACKEND_CORS_ORIGINS=http://localhost:5173,http://localhost:3000
"""

with open("/content/magiv2/.env", "w") as f:
    f.write(env_content)

print("✅ .env file created!")
print(f"   LLM: {llm_model}")
print(f"   Base URL: {llm_base_url}")

## 🐘 BƯỚC 3: Khởi động PostgreSQL trên Colab

In [ ]:
%%bash
# Cài và khởi động PostgreSQL trực tiếp trên Colab (không cần Docker)
apt-get install -qq postgresql postgresql-contrib > /dev/null 2>&1

# Khởi động service
service postgresql start

echo "✅ PostgreSQL started!"

In [ ]:
import subprocess

# Tạo user và database
commands = [
    f"sudo -u postgres psql -c \"CREATE USER {POSTGRES_USER} WITH PASSWORD '{POSTGRES_PASSWORD}';\"",
    f"sudo -u postgres psql -c \"CREATE DATABASE {POSTGRES_DB} OWNER {POSTGRES_USER};\"",
    f"sudo -u postgres psql -c \"GRANT ALL PRIVILEGES ON DATABASE {POSTGRES_DB} TO {POSTGRES_USER};\"",
]

for cmd in commands:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ {cmd.split('psql -c')[1][:50]}...")
    else:
        print(f"⚠️ {result.stderr.strip()[:100]}")

print("\n✅ Database ready!")

In [ ]:
# Chạy Alembic migrations để tạo tables
%cd /content/magiv2
!python -m alembic upgrade head
print("✅ Database migrations done!")

## 🚀 BƯỚC 4: Khởi động FastAPI Server + Public URL (ngrok)

In [ ]:
# Cài ngrok để expose server ra internet
!pip install -q pyngrok

# Lấy token tại https://dashboard.ngrok.com/get-started/your-authtoken (free)
NGROK_TOKEN = ""  # ← ĐIỀN TOKEN NGROK CỦA BẠN

from pyngrok import ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

# Mở tunnel
public_url = ngrok.connect(8008)
print(f"✅ Public URL: {public_url}")
print(f"📖 API Docs: {public_url}/docs")
print(f"\n⚠️ Lưu URL này lại để dùng từ máy tính của bạn!")

In [ ]:
# Khởi động FastAPI server (chạy background)
import subprocess
import threading
import time

os.chdir("/content/magiv2")

def run_server():
    subprocess.run(
        ["python", "-m", "uvicorn", "api.server:app",
         "--host", "0.0.0.0", "--port", "8008"],
        cwd="/content/magiv2"
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(5)  # Chờ server khởi động

# Kiểm tra server đã chạy chưa
import requests
try:
    resp = requests.get("http://localhost:8008/docs")
    print("✅ Server đang chạy!")
    print(f"📖 Truy cập API: {public_url}/docs")
except Exception as e:
    print(f"❌ Server chưa sẵn sàng: {e}")
    print("Thử chờ thêm 10s rồi chạy lại cell này")

## 🧪 BƯỚC 5: Test Pipeline Trực Tiếp (Không cần Frontend)

In [ ]:
# Test: Upload ảnh manga và lấy transcript
import requests
import json
from pathlib import Path

BASE_URL = "http://localhost:8008"

def get_transcript(image_paths: list, story_name: str, chapter_name: str):
    """Gửi ảnh manga để lấy transcript (OCR + detection)"""
    files = [("chapter_pages", (Path(p).name, open(p, "rb"), "image/jpeg"))
             for p in image_paths]
    data = {
        "story_name": story_name,
        "chapter_name": chapter_name,
    }
    response = requests.post(f"{BASE_URL}/main/get-transcript", files=files, data=data)
    return response.json()


def translate_and_inpaint(image_paths: list, story_name: str,
                           chapter_number: int, page_numbers: list,
                           target_lang: str = "vi"):
    """Dịch + inpaint manga"""
    files = [("page_images", (Path(p).name, open(p, "rb"), "image/jpeg"))
             for p in image_paths]
    data = {
        "story_name": story_name,
        "chapter_number": chapter_number,
        "page_numbers": page_numbers,
        "target_lang": target_lang,
    }
    response = requests.post(f"{BASE_URL}/main/translate-and-inpaint",
                             files=files, data=data)
    return response

print("✅ Helper functions ready!")
print("Tiếp theo: Upload ảnh manga vào Colab rồi gọi get_transcript()")

In [ ]:
# ===== DEMO: Chạy với ảnh manga của bạn =====
# Upload ảnh vào Colab trước (kéo thả vào file browser bên trái)

from google.colab import files

print("📂 Upload ảnh manga (JPG/PNG):")
uploaded = files.upload()

uploaded_paths = []
for filename, content in uploaded.items():
    path = f"/content/{filename}"
    with open(path, "wb") as f:
        f.write(content)
    uploaded_paths.append(path)
    print(f"  ✅ {filename}")

print(f"\nTotal: {len(uploaded_paths)} ảnh uploaded")

In [ ]:
# Chạy OCR + Detection
STORY_NAME = "my_manga"    # ← Đổi tên truyện
CHAPTER_NAME = "chapter_1" # ← Đổi tên chapter

print(f"🔍 Đang xử lý {len(uploaded_paths)} trang...")
result = get_transcript(uploaded_paths, STORY_NAME, CHAPTER_NAME)

print("\n📝 Kết quả OCR:")
for i, page_result in enumerate(result):
    print(f"\n--- Trang {i+1} ---")
    transcripts = page_result.get("transcript", [])
    for t in transcripts[:5]:  # Hiện 5 dòng đầu
        print(f"  [{t.get('character', '?')}]: {t.get('text', '')}")

In [ ]:
# Dịch + Inpaint (sau khi đã có transcript)
CHAPTER_NUMBER = 1
PAGE_NUMBERS = list(range(1, len(uploaded_paths) + 1))
TARGET_LANG = "vi"  # Vietnamese

print(f"🌐 Đang dịch và inpaint {len(uploaded_paths)} trang...")
response = translate_and_inpaint(
    image_paths=uploaded_paths,
    story_name=STORY_NAME,
    chapter_number=CHAPTER_NUMBER,
    page_numbers=PAGE_NUMBERS,
    target_lang=TARGET_LANG,
)

if response.status_code == 200:
    # Lưu file zip kết quả
    output_path = f"/content/{STORY_NAME}_{CHAPTER_NAME}_translated.zip"
    with open(output_path, "wb") as f:
        f.write(response.content)
    print(f"✅ Xong! Kết quả lưu tại: {output_path}")

    # Download về máy
    files.download(output_path)
else:
    print(f"❌ Lỗi: {response.status_code}")
    print(response.text[:500])

## 💾 BƯỚC 6: Lưu vào Google Drive (để không mất khi Colab restart)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Tạo thư mục trong Drive
!mkdir -p "/content/drive/MyDrive/MagiV2"

# Backup transcript history
!cp -r /content/magiv2/transcript_history "/content/drive/MyDrive/MagiV2/"

# Backup kết quả dịch
!cp /content/*.zip "/content/drive/MyDrive/MagiV2/" 2>/dev/null || true

print("✅ Đã backup lên Google Drive!")
print("   Xem tại: Drive > MyDrive > MagiV2")

---
## 📋 Ghi chú quan trọng

| Vấn đề | Giải pháp |
|---|---|
| Session Colab hết (~12h free) | Lưu transcript vào Drive trước khi mất |
| Model download lại mỗi lần | Mount Drive và cache HF models vào Drive |
| PostgreSQL reset khi restart | Data chỉ tồn tại trong session, backup thường xuyên |
| Ngrok URL thay đổi mỗi lần | Lấy URL mới sau mỗi lần restart |

### ⚡ Tips để tiết kiệm thời gian:
```python
# Cache HuggingFace models vào Drive để không download lại
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/MagiV2/hf_cache"
# Chạy cell này TRƯỚC KHI import transformers/torch
```